In [1]:
# Instalación silenciosa de dependencias
!pip install -qU streamlit langchain-classic langchain-groq langchain-chroma langchain-huggingface sentence-transformers pyngrok

In [2]:
!cp /content/drive/MyDrive/Proyecto_RAG_Coyuntura/motor_rag.py .

In [3]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("Variable de entorno GROQ_API_KEY configurada correctamente.")
except Exception as e:
    print("Error: La clave 'GROQ_API_KEY' no se encuentra definida en los Secrets del entorno.")

try:
    NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    print("Token de ngrok cargado correctamente.")
except Exception as e:
    print("Error: La clave 'NGROK_AUTH_TOKEN' no se encuentra definida en los Secrets del entorno.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Variable de entorno GROQ_API_KEY configurada correctamente.
Token de ngrok cargado correctamente.


In [4]:
%%writefile app.py
import streamlit as st
import os
from motor_rag import PipelineCoyuntura

DB_DIR = '/content/drive/MyDrive/Proyecto_RAG_Coyuntura/chroma_db'

st.set_page_config(page_title="RAG Coyuntura Bolivia", layout="wide")
st.title("Análisis de Coyuntura - Corpus: urgente.bo")

@st.cache_resource
def iniciar_motor():
    pipeline = PipelineCoyuntura(directorio_db=DB_DIR)
    pipeline.cargar_motor()
    return pipeline

try:
    motor = iniciar_motor()
except Exception as e:
    st.error(f"Error en la inicialización del motor RAG. Detalle técnico: {e}")
    st.stop()

# Parámetros de filtrado
meses_disponibles = ["Todos", "enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
mes_seleccionado = st.selectbox("Restringir búsqueda por mes (Opcional):", meses_disponibles)

pregunta = st.text_area("Ingrese su consulta analítica:")

if st.button("Ejecutar Análisis") and pregunta:
    with st.spinner("Procesando análisis y recuperando documentos..."):

        filtro_aplicable = mes_seleccionado if mes_seleccionado != "Todos" else None
        resultado = motor.consultar(pregunta, mes_filtro=filtro_aplicable)

        st.markdown("### Resultado del Análisis")
        st.write(resultado["answer"])

        with st.expander("Desglose de fragmentos recuperados (Auditoría)"):
            for i, doc in enumerate(resultado["context"]):
                st.markdown(f"**Fuente {i+1}:** {doc.metadata.get('titulo', 'Sin título')}")
                st.markdown(f"- **Fecha:** {doc.metadata.get('fecha', 'N/D')} | **URL:** {doc.metadata.get('url', 'N/D')}")
                st.info(doc.page_content[:350] + "...")

Overwriting app.py


In [5]:
from pyngrok import ngrok
import time

# Finalización de procesos previos para evitar conflictos de puertos
!pkill -f streamlit
ngrok.kill()  # cierra túneles de ngrok previos, si los hay

# Autenticación de ngrok
ngrok.set_auth_token(NGROK_TOKEN)

# Ejecución de Streamlit en segundo plano
!streamlit run app.py &>/content/logs.txt &
time.sleep(5)

# Verificación de que streamlit arrancó bien antes de abrir el túnel
!tail -n 20 /content/logs.txt

# Exposición del puerto 8501
public_url = ngrok.connect(8501)
print(f"\nAcceda a su aplicación en:\n{public_url}")




Acceda a su aplicación en:
NgrokTunnel: "https://skipper-smartness-hazily.ngrok-free.dev" -> "http://localhost:8501"
